# Task 02 — Advanced Data Cleaning & Preprocessing

**Student:** Abdul Haseeb ([@ahaseeb003](https://github.com/ahaseeb003))
**Program:** 2-Month Machine Learning Internship — Month 1

**Dataset:** NYC Airbnb Open Data 2019 (`AB_NYC_2019.csv`), loaded from Google Drive.

This notebook builds a reusable, production-grade data cleaning pipeline that handles:
- Missing values
- Outliers
- Type inference
- Schema validation


## 1. Setup & Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_PATH = "/content/drive/MyDrive/dataset/AB_NYC_2019.csv"
OUTPUT_DIR = "/content/drive/MyDrive/Task_02_Advanced_Data_Cleaning_and_Preprocessing"
os.makedirs(OUTPUT_DIR, exist_ok=True)

assert os.path.exists(DATA_PATH), f"File not found: {DATA_PATH}. Check the path in your Drive."
print("Dataset found at:", DATA_PATH)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
pd.set_option("display.max_columns", None)


## 2. Load the Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
df.describe(include="all").T


## 3. Exploratory Data Analysis (Before Cleaning)

The AB_NYC_2019 dataset is a real, messy dataset with well-known problems:
- `name` and `host_name` have a few missing values
- `last_review` and `reviews_per_month` are missing whenever a listing has **zero reviews**
  (this is not random — it's a structural pattern, not truly "missing")
- `price` contains a small number of `0` values, which is not a realistic nightly rate
- `minimum_nights` has extreme outliers (some listings require 1000+ nights)
- Latitude/longitude should fall within NYC's geographic bounds


In [ ]:
# Missing values per column
missing_summary = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing_summary / len(df) * 100).round(2)
missing_report = pd.DataFrame({"missing_count": missing_summary, "missing_pct": missing_pct})
missing_report[missing_report["missing_count"] > 0]


In [ ]:
# Visualization 1: Missing values heatmap
plt.figure(figsize=(9, 5))
sns.heatmap(df.isna(), cbar=False, cmap="viridis")
plt.title("Missing Value Pattern (Raw Data)")
plt.tight_layout()
plt.show()


In [ ]:
# Visualization 2: Missing value counts bar chart
plt.figure(figsize=(8, 5))
missing_report[missing_report["missing_count"] > 0]["missing_count"].plot(kind="bar", color="indianred")
plt.title("Missing Values per Column (Raw Data)")
plt.ylabel("Number of Missing Values")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Listings with price == 0:", (df["price"] == 0).sum())
print("Max minimum_nights:", df["minimum_nights"].max())
print("Rows where number_of_reviews == 0 but reviews_per_month is missing:",
      ((df["number_of_reviews"] == 0) & (df["reviews_per_month"].isna())).sum())


In [ ]:
# Visualization 3: Price distribution (raw) — outliers clearly visible
plt.figure(figsize=(7, 5))
sns.boxplot(x=df["price"], color="lightblue")
plt.title("Price Distribution (Raw Data) — Outliers Visible")
plt.tight_layout()
plt.show()


In [ ]:
# Visualization 4: minimum_nights distribution (raw) — extreme outliers
plt.figure(figsize=(7, 5))
sns.boxplot(x=df["minimum_nights"], color="lightcoral")
plt.title("Minimum Nights (Raw Data) — Extreme Outliers Visible")
plt.tight_layout()
plt.show()


## 4. Building the Cleaning Pipeline

The pipeline is built as small, well-documented, reusable functions — each does exactly
one job, which makes it easy to test and review.


In [ ]:
def fix_types(data: pd.DataFrame) -> pd.DataFrame:
    """Coerce columns to their correct data types.

    - 'last_review' -> real datetime
    - 'neighbourhood_group', 'neighbourhood', 'room_type' -> pandas 'category' dtype
    """
    data = data.copy()
    data["last_review"] = pd.to_datetime(data["last_review"], errors="coerce")

    for col in ["neighbourhood_group", "neighbourhood", "room_type"]:
        data[col] = data[col].astype("category")

    return data


In [ ]:
def validate_schema(data: pd.DataFrame) -> pd.DataFrame:
    """Enforce realistic value ranges (schema validation).

    Values outside a sane business range are treated as invalid and set to NaN,
    so they're handled consistently by the missing-value step later.
    """
    data = data.copy()

    # Price of 0 (or negative) is not a real nightly rate
    data.loc[data["price"] <= 0, "price"] = np.nan

    # minimum_nights must be at least 1
    data.loc[data["minimum_nights"] < 1, "minimum_nights"] = np.nan

    # Latitude/longitude must fall within NYC's rough geographic bounding box
    nyc_lat_range = (40.4, 41.0)
    nyc_lon_range = (-74.3, -73.6)
    bad_coords = (
        ~data["latitude"].between(*nyc_lat_range)
        | ~data["longitude"].between(*nyc_lon_range)
    )
    data.loc[bad_coords, ["latitude", "longitude"]] = np.nan

    return data


In [ ]:
def handle_structural_missingness(data: pd.DataFrame) -> pd.DataFrame:
    """Handle the 'missing but meaningful' pattern in this dataset.

    When a listing has zero reviews, 'reviews_per_month' and 'last_review' are
    missing NOT at random -- it simply means the listing has never been reviewed.
    We encode that explicitly instead of treating it as ordinary missing data.
    """
    data = data.copy()
    no_reviews_mask = data["number_of_reviews"] == 0

    data.loc[no_reviews_mask, "reviews_per_month"] = 0.0
    data["has_reviews"] = ~no_reviews_mask

    return data


In [ ]:
def handle_outliers(data: pd.DataFrame, columns, factor=1.5) -> pd.DataFrame:
    """Cap outliers using the IQR method (Winsorization) instead of dropping rows,
    so we don't lose valuable listings.
    """
    data = data.copy()

    for col in columns:
        q1 = data[col].quantile(0.25)
        q3 = data[col].quantile(0.75)
        iqr = q3 - q1
        lower = max(q1 - factor * iqr, data[col].min())
        upper = q3 + factor * iqr
        data[col] = data[col].clip(lower=lower, upper=upper)

    return data


In [ ]:
def handle_missing_values(data: pd.DataFrame) -> pd.DataFrame:
    """Impute remaining missing values with sensible, column-appropriate strategies.

    - 'name' / 'host_name' -> 'Unknown' (text placeholders, not meaningful to model)
    - Numeric columns (price, coordinates, minimum_nights) -> median (robust to outliers)
    - 'last_review' is left as NaT -- a missing date for a never-reviewed listing is
      correct information, not an error, so we do not fabricate a date.
    """
    data = data.copy()

    for col in ["name", "host_name"]:
        data[col] = data[col].fillna("Unknown")

    numeric_cols = ["price", "minimum_nights", "latitude", "longitude"]
    for col in numeric_cols:
        data[col] = data[col].fillna(data[col].median())

    return data


In [ ]:
def remove_duplicates(data: pd.DataFrame) -> pd.DataFrame:
    """Drop exact duplicate rows, keeping the first occurrence."""
    return data.drop_duplicates().reset_index(drop=True)


In [ ]:
def clean_pipeline(data: pd.DataFrame) -> pd.DataFrame:
    """Run the full cleaning pipeline end-to-end, in the right order:

    1. Fix data types
    2. Validate schema (turn impossible values into NaN)
    3. Remove exact duplicates
    4. Handle the structural (meaningful) missingness pattern
    5. Cap outliers
    6. Impute any remaining missing values
    """
    data = fix_types(data)
    data = validate_schema(data)
    data = remove_duplicates(data)
    data = handle_structural_missingness(data)
    data = handle_outliers(data, columns=["price", "minimum_nights"])
    data = handle_missing_values(data)
    return data


## 5. Run the Pipeline

In [ ]:
cleaned_df = clean_pipeline(df)

print("Raw shape:    ", df.shape)
print("Cleaned shape:", cleaned_df.shape)
cleaned_df.head()


In [ ]:
# Sanity checks
print("Remaining missing values:\n", cleaned_df.isna().sum())
print("\nRemaining duplicates:", cleaned_df.duplicated().sum())
print("\nListings with price <= 0:", (cleaned_df["price"] <= 0).sum())
print("\nData types:\n", cleaned_df.dtypes)


## 6. Visualizations — Before vs. After Cleaning

In [ ]:
# Visualization 5: Price distribution before vs after cleaning
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(x=df["price"], ax=axes[0], color="lightcoral")
axes[0].set_title("Price — Before Cleaning")

sns.boxplot(x=cleaned_df["price"], ax=axes[1], color="lightgreen")
axes[1].set_title("Price — After Cleaning")

plt.tight_layout()
plt.show()


In [ ]:
# Visualization 6: minimum_nights before vs after cleaning
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(x=df["minimum_nights"], ax=axes[0], color="salmon")
axes[0].set_title("Minimum Nights — Before Cleaning")

sns.boxplot(x=cleaned_df["minimum_nights"], ax=axes[1], color="mediumseagreen")
axes[1].set_title("Minimum Nights — After Cleaning")

plt.tight_layout()
plt.show()


In [ ]:
# Visualization 7: Missing values heatmap after cleaning (should be empty except last_review)
plt.figure(figsize=(9, 3))
sns.heatmap(cleaned_df.isna(), cbar=False, cmap="viridis")
plt.title("Missing Value Pattern (Cleaned Data)")
plt.tight_layout()
plt.show()


In [ ]:
# Visualization 8: Average price by neighbourhood group (cleaned data)
plt.figure(figsize=(8, 5))
sns.boxplot(data=cleaned_df, x="neighbourhood_group", y="price", palette="pastel")
plt.title("Price by Neighbourhood Group (Cleaned Data)")
plt.tight_layout()
plt.show()


In [ ]:
# Visualization 9: Listing locations colored by room type (cleaned data)
plt.figure(figsize=(8, 7))
sns.scatterplot(
    data=cleaned_df, x="longitude", y="latitude", hue="room_type",
    s=8, alpha=0.4, palette="viridis"
)
plt.title("Airbnb Listings Across NYC by Room Type (Cleaned Data)")
plt.tight_layout()
plt.show()


## 7. Save the Cleaned Dataset

In [ ]:
cleaned_path = os.path.join(OUTPUT_DIR, "AB_NYC_2019_cleaned.csv")
cleaned_df.to_csv(cleaned_path, index=False)
print("Cleaned data saved to:", cleaned_path)


## 8. Summary of Decisions

| Problem | Decision | Why |
|---|---|---|
| `price == 0` | Converted to NaN, then imputed with **median** | A free nightly rate isn't realistic; median is robust to the dataset's heavy right skew |
| `minimum_nights` extreme outliers (1000+) | **Capped (Winsorized)** using IQR method | Keeps every listing's row instead of dropping data |
| `reviews_per_month` missing when `number_of_reviews == 0` | Filled with **0**, plus new `has_reviews` flag | This missingness is structural/meaningful, not random — encoding it explicitly preserves the signal |
| `last_review` missing | Left as `NaT`, **not imputed** | A missing review date for a never-reviewed listing is correct information; fabricating a date would be misleading |
| `name` / `host_name` missing | Filled with **'Unknown'** | Text fields not used numerically; placeholder avoids dropping rows |
| Invalid latitude/longitude (outside NYC bounds) | Converted to NaN, then imputed with **median** | Keeps coordinates realistic for any downstream mapping/geo analysis |
| Duplicate rows | **Dropped**, kept first occurrence | Avoids double-counting listings |

**Next step:** see the companion notebook `Task_02_Bonus_Challenge.ipynb`, which wraps this
exact pipeline into a scikit-learn compatible `Transformer` class.
